## 0. This checkout, not whatever is installed

`zou_lab_control_v2` is the one entry: importing it puts this checkout's eight layers
on the path, ahead of anything else.  It has to come **first** -- if a `zlc_*` module
was already imported from somewhere else, it refuses out loud rather than leaving two
copies in one kernel.  (If that happens: restart the kernel and run this cell first.)


In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
while not (_here / 'zou_lab_control_v2').is_dir() and _here != _here.parent:
    _here = _here.parent
sys.path.insert(0, str(_here))

import zou_lab_control_v2

print('code in use:', zou_lab_control_v2.ROOT)


# zlc_ui：完整 GUI 示例与外部 API 验收

这个 Notebook 把每个 GUI 都作为独立、可直接运行的 cell 展示：Gallery、TaskConsole、PulseEditor、FigureViewer、DeviceManager。每个 cell 都直接调用对应的 `create_window()`；不会把其他 GUI 藏在 TaskConsole 的 cell 或一个不可见的内部循环里。

所有数据都是 fake data。Notebook 使用真实 Qt 窗口和唯一的 `capture_window()` 验收入口，不创建固定尺寸 oracle，也不做截图差分。

## 0. 安装与 Qt 生命周期

~~~powershell
python -m pip install -e ".[dev]"
python -m pip install jupyter ipykernel
jupyter notebook notebooks/usage.ipynb
~~~

新 kernel 中不要先运行 `%gui qt`，也不要在代码格调用 `app.exec_()`。`ensure_qt_app()` 是唯一的 QApplication 入口；Notebook 由 IPython 继续泵 Qt 事件。视觉验收必须在真实桌面运行，`offscreen` 只用于对象级 smoke test。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("请从 zlc_ui 仓库或 notebooks/ 子目录启动这个 Notebook")
for import_root in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

from zlc_ui import WINDOW_SCREEN_FRACTION, __version__, capture_window, ensure_qt_app

app = ensure_qt_app(["zlc-ui-usage-notebook"])
print("project root:", PROJECT_ROOT)
print("zlc_ui version:", __version__)
print("Qt platform:", app.platformName())
print("shared screen fraction:", WINDOW_SCREEN_FRACTION)

DEMO_WINDOW_NAMES = (
    "gallery_window", "console_window", "pulse_window",
    "figure_window", "device_window", "api_window",
)

def close_demo_windows():
    """Close every Notebook-owned window before the next demo cell."""
    for name in DEMO_WINDOW_NAMES:
        window = globals().get(name)
        if window is not None:
            window.close()
            window.deleteLater()
            globals()[name] = None
    app.processEvents()


### 0.1 顶层 facade：可复用的无领域 API

下面的名字都从 `zlc_ui` 顶层导入。`FormSpec` 是有序字段投影，`FormChoice` 是带类型值的选项，`FormRuntimeContext` 注入动态选项，`BoardMetrics` 注入板级几何策略；它们不拥有业务状态。

In [ ]:
from zlc_ui import (
    BoardMetrics,
    FormChoice,
    FormFieldProps,
    FormRuntimeContext,
    FormSpec,
)

mode_choices = (FormChoice("Preview", "preview"), FormChoice("Live", "live"))
facade_spec = FormSpec((
    FormFieldProps("mode", "choice", "Mode", default="preview", choices=mode_choices),
    FormFieldProps("count", "int", "Count", default=2, minimum=0, maximum=9),
))
facade_runtime = FormRuntimeContext(choice_names=lambda key: ("preview", "live") if key == "mode" else ())
facade_metrics = BoardMetrics(12, lambda _size: (320, 240))
print("facade form keys/defaults:", facade_spec.keys, facade_spec.default_values())
print("dynamic mode choices:", facade_runtime.names_for("mode"))
print("board metrics:", facade_metrics.gap, facade_metrics.card_size("1x1"))


## 1. Gallery：基础组件 → 组合件 → 完整 GUI

Gallery 是总览入口：先看带名称的基础组件，再看带名称的组合件（包括 Scan slot / API slot），最后在完整 GUI tab 中切换四套正式 view。Scan/API 示例不是静态样式：请点击输入框右侧圆点，观察 duration 从 off → Scan → API → off，delay 从 off → API → off。这个 cell 只负责打开 Gallery。

In [ ]:
from examples.gallery import create_window as create_gallery_window

close_demo_windows()
gallery_window = create_gallery_window()
app.processEvents()
print("Gallery visible:", gallery_window.isVisible())

## 2. 每个完整 GUI 的独立运行 cell

下面四个 cell 不依赖 Gallery tab，也不依赖前一个 GUI。每格先关闭 Notebook 自己打开的旧窗口，再直接调用正式 demo 的 `create_window()`，因此可以单独运行、重复运行和逐个截图验收。

### 2.1 TaskConsole

正式入口：`examples.demo_console.create_window()`。这个 demo 展示混合 `1x4 / 2x2 / 4x2` card、逻辑行、状态条和对外信号回显。

In [ ]:
from examples.demo_console import create_window as create_console_window

close_demo_windows()
console_window = create_console_window()
app.processEvents()
print("TaskConsole visible:", console_window.isVisible())

### 2.2 PulseEditor

正式入口:`zlc_ui.open_pulse_editor()`。它**返回一个句柄**——信号 + `set_*`/`show_*`,没有任何 widget:
外部只负责接线和逻辑,窗口本身(无边框 chrome、屏幕适配尺寸、居中、关闭握手)归 zlc_ui。
`examples.demo_pulse_editor.create_window()` 就是这样开的,并用下面这套 VM 词汇把假数据喂进去。


In [ ]:
from zlc_ui import (
    DelayRowVM,
    FieldVM,
    PeriodVM,
    PortRowVM,
    RepeatVM,
    ScanPageRecord,
    ScheduleVM,
    TargetPortRecord,
    TargetWidthRule,
    VALIDATOR_FLOAT,
    VALIDATOR_INT,
    open_pulse_editor,
)
from examples.demo_pulse_editor import create_window as create_pulse_window

close_demo_windows()
pulse_editor = create_pulse_window()
print("handle:", type(pulse_editor).__name__)
print("window:", pulse_editor.window_title(), pulse_editor.window_size())
# Everything the outside may do is on the handle; there is no page to reach.
print("port:", len([n for n in dir(pulse_editor) if not n.startswith('_')]), "members")
demo_windows.append(pulse_editor)


#### Pulse binding 的 presenter API

Qt 圆点只发出 binding_cycle_requested(field_kind, period_id, port_key)；外部 presenter 持有 FieldVM，用下面的纯 helper 决定下一态，再调用 set_period() 或 set_delay_row() 回投影。

In [ ]:
from zlc_ui import cycle_binding_kind

duration_binding = None
duration_trace = []
for _ in range(3):
    duration_binding = cycle_binding_kind(duration_binding, field_kind="duration")
    duration_trace.append(duration_binding or "off")

delay_binding = None
delay_trace = []
for _ in range(2):
    delay_binding = cycle_binding_kind(delay_binding, field_kind="delay")
    delay_trace.append(delay_binding or "off")
print("duration:", duration_trace)
print("delay:", delay_trace)


### 2.3 FigureViewer

正式入口：`examples.demo_figure_viewer.create_window()`。这个 demo 展示 File / Browse、InfoPane tabs 和 presenter-owned figure QWidget 挂载位。

In [ ]:
from examples.demo_figure_viewer import create_window as create_figure_window

close_demo_windows()
figure_window = create_figure_window()
app.processEvents()
print("FigureViewer visible:", figure_window.isVisible())

### 2.4 DeviceManager

正式入口：`examples.demo_device_manager.create_window()`。这个 demo 展示 fake sensor / camera、role/type 选择、Count / Enabled FormSpec 和状态条。

In [ ]:
from examples.demo_device_manager import create_window as create_device_window

close_demo_windows()
device_window = create_device_window()
app.processEvents()
print("DeviceManager visible:", device_window.isVisible())

## 3. 外部应用如何调用 view API

外部 presenter 持有业务状态，view 只接收 plain values、挂载 host-owned `QWidget`，并通过 `*_requested` / `*_committed` 信号把用户意图送回外部。下面是一个独立的 TaskConsole 组装例，不是上面完整 demo 的替代品。

In [ ]:
from zlc_ui import open_task_console
from examples.demo_console import create_window as create_console_window

close_demo_windows()
console = create_console_window()
print("handle:", type(console).__name__, "|", console.window_title())
# A host says which panels exist and what each shows; the cards, their
# controls and the board layout are all on the other side of the wall.
console.set_summary("driven from the notebook")
console.set_panel_order(("panel-3", "panel-1", "panel-2"))
print("panels:", console.panel_ids())
demo_windows.append(console)


### 3.1 外部 DeviceManager API

DeviceManager 同样由外部提供 choices、device records 和 `FormSpec`；view 不拥有 catalog、持久化或设备后端。

In [ ]:
from zlc_ui import FormChoice, FormFieldProps, FormSpec, open_device_manager
from examples.demo_device_manager import create_window as create_device_window

close_demo_windows()
devices = create_device_window()
print("handle:", type(devices).__name__, "|", devices.window_title())
# The host supplies choices, records and the FormSpec; the cards, their
# forms and the status line are all on the other side of the wall.
devices.set_devices((("sensor-1", "input", "sensor"), ("dac-1", "bias", "dac")))
devices.show_status("two fake devices", "idle")
print("read back:", devices.read_values("sensor-1"))
demo_windows.append(devices)


## 4. 唯一的 UI 验收 API：真实屏幕比例

下面的 cell 不是 GUI 示例，而是批量验收入口。它仍然调用上面五个正式 `create_window()` factory；`capture_window()` 负责真实屏幕尺寸、物理 DPR、标题栏/内容边界和窗口抓图。offscreen 会明确拒绝。默认不写文件；需要保存证据时，再把 `capture_root` 改成你指定的目录。

In [ ]:
from examples.demo_console import create_window as create_console_window
from examples.demo_device_manager import create_window as create_device_window
from examples.demo_figure_viewer import create_window as create_figure_window
from examples.demo_pulse_editor import create_window as create_pulse_window
from examples.gallery import create_window as create_gallery_window

factories = {
    "gallery": create_gallery_window,
    "console": create_console_window,
    "pulse": create_pulse_window,
    "figure": create_figure_window,
    "device": create_device_window,
}
capture_root = None  # 例如：PROJECT_ROOT / "ui-captures"
if app.platformName().strip().lower() == "offscreen":
    try:
        capture_window(create_gallery_window)
    except RuntimeError as error:
        print("real-screen acceptance intentionally skipped:", error)
else:
    reports = {}
    for name, factory in factories.items():
        options = {} if capture_root is None else {
            "output": capture_root / f"{name}-window.png",
            "desktop_output": capture_root / f"{name}-screen.png",
        }
        reports[name] = capture_window(factory, **options)


## 5. 独立 Python 应用的最小生命周期

~~~python
from zlc_ui import ensure_qt_app, open_device_manager

app = ensure_qt_app(["my-app"])
devices = open_device_manager(title="my-app")
devices.show_status("host-owned status", "idle")
app.exec_()
~~~

一个 GUI = 一次调用 + 一个句柄。宿主接线和喂数据,窗口本身(无边框 chrome、屏幕适配尺寸、居中、
关闭握手)全在 zlc_ui 里;宿主拿不到任何 widget,也就无从自己攒一套界面。
普通 Python 程序由宿主调用 `app.exec_()`;Notebook 不调用它。


In [ ]:
close_demo_windows()
print("Notebook windows closed")